In [0]:
customers_df = spark.read.table("processed_db.customers")

###Customer Count by City

In [0]:
from pyspark.sql.functions import countDistinct, col

customers_by_city_df = (
    customers_df
        .groupBy("city")
        .agg(countDistinct("customer_id").alias("customers"))
        .orderBy(col("customers").desc())
)

customers_by_city_df.show(truncate=False)

+---------+---------+
|city     |customers|
+---------+---------+
|Bangalore|2        |
|Mumbai   |2        |
|Delhi    |2        |
|Chennai  |1        |
|NULL     |1        |
|Kolkata  |1        |
|Pune     |1        |
+---------+---------+



###Customer Count by Age Group

In [0]:
customers_by_age_group_df = (
    customers_df
        .groupBy("age_group")
        .agg(countDistinct("customer_id").alias("customers"))
        .orderBy(col("customers").desc())
)

customers_by_age_group_df.show(truncate=False)

+---------+---------+
|age_group|customers|
+---------+---------+
|26-35    |5        |
|36-45    |3        |
|46+      |1        |
|Unknown  |1        |
+---------+---------+



###gender distribution

In [0]:
from pyspark.sql.functions import round as spark_round, lit

total_customers = customers_df.select(countDistinct("customer_id").alias("n")).collect()[0]["n"]

gender_dist_df = (
    customers_df
        .groupBy("gender")
        .agg(countDistinct("customer_id").alias("customers"))
        .withColumn("pct", spark_round((col("customers") / lit(total_customers)) * 100, 2))
        .orderBy(col("customers").desc())
)

gender_dist_df.show(truncate=False)

+------+---------+----+
|gender|customers|pct |
+------+---------+----+
|F     |5        |50.0|
|M     |5        |50.0|
+------+---------+----+



###Save the output to presentation container as delta table

In [0]:
customers_by_city_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customers_by_city")

In [0]:
customers_by_age_group_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customers_by_age")

In [0]:
gender_dist_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("presentation_db.customers_gender")